In [2]:
suppressPackageStartupMessages({
    library(DESeq2)
    library(tidyverse)
})

dir_proc <- "../data/processed"
dir_tablas <- "../results/tablas"

conteos <- read.csv(file.path(dir_proc, "pseudobulk_T_conteos.csv"), row.names = 1, check.names = FALSE)
meta <- read.csv(file.path(dir_proc, "pseudobulk_T_meta.csv"), row.names = 1)
stopifnot(identical(colnames(conteos), rownames(meta)))
meta$condicion <- factor(meta$condicion, levels = c("Sano", "NL", "LS"))
meta$donante <- factor(meta$donante)

genes_y <- c("RPS4Y1", "DDX3Y", "UTY", "KDM5D", "USP9Y", "EIF1AY")
meta$sexo <- factor(ifelse(colSums(conteos[genes_y, ]) > 20, "M", "F"))

tecnicos <- grepl("^(MT-|MTRNR2L|RP[SL]|TR[ABDG][VJ]|IG[HKL][VJ])", rownames(conteos))
conteos <- conteos[!tecnicos, ]

In [3]:
correr_de <- function(excluir = character(0), diseno = ~ condicion) {
    m <- meta[!rownames(meta) %in% excluir, ]
    dds <- DESeqDataSetFromMatrix(as.matrix(conteos[, rownames(m)]), colData = m, design = diseno)
    dds <- dds[rowSums(counts(dds) >= 10) >= 4, ]
    DESeq(dds, quiet = TRUE)
}

q_de <- function(res, gen) if (gen %in% rownames(res)) res[gen, "padj"] else NA

resumir <- function(dds, nombre) {
    map_dfr(c("NL", "LS"), function(nivel) {
        res <- results(dds, contrast = c("condicion", nivel, "Sano"), alpha = 0.1)
        tibble(escenario = nombre,
               contraste = paste(nivel, "vs Sano"),
               genes = nrow(res),
               q_menor_0.1 = sum(res$padj < 0.1, na.rm = TRUE),
               top_5 = paste(head(rownames(res)[order(res$pvalue)], 5), collapse = ", "),
               q_FXYD2 = q_de(res, "FXYD2"),
               q_ZNF683 = q_de(res, "ZNF683"))
    })
}

In [4]:
escenarios <- list(
    "Original" = character(0),
    "Sin CON194" = "CON194",
    "Sin DER114_LS y DER115_LS" = c("DER114_LS", "DER115_LS")
)

sens <- imap_dfr(escenarios, ~ resumir(correr_de(.x), .y))
sens <- bind_rows(sens, resumir(correr_de(diseno = ~ sexo + condicion), "Con sexo en el modelo"))

write.csv(sens, file.path(dir_tablas, "sensibilidad_de_T.csv"), row.names = FALSE)
sens

escenario,contraste,genes,q_menor_0.1,top_5,q_FXYD2,q_ZNF683
<chr>,<chr>,<int>,<int>,<chr>,<dbl>,<dbl>
Original,NL vs Sano,3521,1,"FXYD2, ZNF683, AL133415.1, STX17-AS1, JMY",0.02876865,0.1963048
Original,LS vs Sano,3521,0,"PTPRCAP, STX17-AS1, ATF3, FXYD2, ID3",0.99998563,0.9999856
Sin CON194,NL vs Sano,3519,1,"FXYD2, ZNF683, TNFAIP3, AL133415.1, RCBTB2",0.02392947,0.3410016
Sin CON194,LS vs Sano,3519,0,"PTPRCAP, STX17-AS1, FXYD2, ID3, AL627171.2",0.99801277,0.9980128
Sin DER114_LS y DER115_LS,NL vs Sano,2259,1,"FXYD2, ZNF683, AL133415.1, NEU1, TNFAIP3",0.02189797,0.1075377
Sin DER114_LS y DER115_LS,LS vs Sano,2259,0,"GLIPR1, ZNF683, RNF187, BIN1, MYDGF",0.99174140,0.9917414
Con sexo en el modelo,NL vs Sano,3521,0,"FXYD2, ZNF683, PTPRCAP, AL133415.1, STX17-AS1",0.11065680,0.9563134
Con sexo en el modelo,LS vs Sano,3521,1,"PTPRCAP, STX17-AS1, ID3, AL627171.2, MRPS6",0.99873916,0.9987392


In [5]:
pares <- meta |>
    rownames_to_column("muestra") |>
    filter(condicion != "Sano") |>
    group_by(donante) |>
    filter(n() == 2) |>
    pull(muestra)
pares

m <- droplevels(meta[pares, ])
dds_par <- DESeqDataSetFromMatrix(as.matrix(conteos[, pares]), colData = m,
                                  design = ~ donante + condicion)
dds_par <- dds_par[rowSums(counts(dds_par) >= 10) >= 3, ]
dds_par <- DESeq(dds_par, quiet = TRUE)

res_par <- results(dds_par, contrast = c("condicion", "LS", "NL"), alpha = 0.1) |>
    as.data.frame() |>
    rownames_to_column("gen") |>
    arrange(pvalue)
write.csv(res_par, file.path(dir_tablas, "de_T_LS_vs_NL_pareado.csv"), row.names = FALSE)

cat("Genes con q < 0.1:", sum(res_par$padj < 0.1, na.rm = TRUE), "\n")
head(res_par, 15)

[1] "DER114_LS" "DER114_NL" "DER115_LS" "DER115_NL" "DER119_LS" "DER119_NL"

Genes con q < 0.1: 0 


,gen,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,ATF3,29.350132,2.331457,0.6319287,3.689431,0.0002247563,0.9997536
2,HSPA1B,507.843665,1.724378,0.5212218,3.308339,0.0009385110,0.9997536
3,RGCC,402.568402,1.446948,0.4528359,3.195303,0.0013968417,0.9997536
4,TNFAIP3,87.303013,1.546706,0.5093738,3.036486,0.0023935344,0.9997536
5,NR4A1,165.335248,1.341531,0.4481358,2.993582,0.0027572377,0.9997536
6,DNAJB1,351.714345,1.525725,0.5168559,2.951935,0.0031578893,0.9997536
7,DUSP2,159.329139,1.339562,0.4697367,2.851730,0.0043481976,0.9997536
8,HSP90AA1,428.868414,1.260167,0.4444759,2.835174,0.0045800698,0.9997536
9,SERTAD1,65.339040,1.533822,0.5623289,2.727624,0.0063792194,0.9997536


In [6]:
genes_ifn <- c("STAT1", "IRF1", "GBP1", "GBP2", "GBP4", "GBP5", "IFI6", "ISG15", "MX1")
res_par |> filter(gen %in% genes_ifn)

gen,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
IFI6,12.46399,-0.69481258,0.6467192,-1.07436516,0.2826591,0.9997536
GBP5,19.31495,0.54407781,0.5953182,0.91392772,0.3607548,0.9997536
ISG15,20.44215,0.47175711,0.5421982,0.87008236,0.3842554,0.9997536
GBP2,19.84056,0.48753311,0.5724853,0.85160810,0.3944316,0.9997536
IRF1,36.88637,0.33278316,0.4860106,0.68472403,0.4935181,0.9997536
STAT1,17.80646,0.12467259,0.5838110,0.21354955,0.8308983,0.9997536
GBP1,9.26211,0.06084489,0.7210970,0.08437824,0.9327557,0.9997536
